
**فصل دوم: ماشین لرنینگ روی صدا**

**درس 6:**

# 🧩 ورود به دنیای احساسات   


### 1️⃣ نصب و وارد کردن کتابخانه‌ها


In [71]:

%pip install -v -i https://mirror-pypi.runflare.com/simple librosa soundfile matplotlib numpy pandas scikit-learn --trusted-host mirror-pypi.runflare.com

Using pip 26.1.1 from c:\Users\mosta\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip (python 3.12)
Looking in indexes: https://mirror-pypi.runflare.com/simple
Note: you may need to restart the kernel to use updated packages.


In [72]:
import os
import time
import numpy as np
import librosa
import sounddevice as sd 
import joblib        
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

### 2️⃣ RAVDESS تنظیم مسیر دیتاست و شناخت ساختار 


**این دیتاست را از لینک زیر میتوانید دانلود کنید**

https://github.com/ZenvilleErasmus/RAVDESS-emotions-speech-audio-only

In [73]:
DATA_DIR = "RAVDESS"

# بررسی پوشه‌های بازیگران
actor_folders = sorted(os.listdir(DATA_DIR))
print(f"تعداد بازیگران: {len(actor_folders)}")
print(f"پوشه‌های نمونه: {actor_folders[:3]}")

# نگاهی به فایل‌های یکی از بازیگران
sample_actor = actor_folders[0]
sample_files = os.listdir(os.path.join(DATA_DIR, sample_actor))
print(f"\nفایل‌های داخل {sample_actor}:")
for f in sample_files[:5]:
    print(f"  {f}")

تعداد بازیگران: 24
پوشه‌های نمونه: ['Actor_01', 'Actor_02', 'Actor_03']

فایل‌های داخل Actor_01:
  03-01-01-01-01-01-01.wav
  03-01-01-01-01-02-01.wav
  03-01-01-01-02-01-01.wav
  03-01-01-01-02-02-01.wav
  03-01-02-01-01-01-01.wav


### 3️⃣ MFCC تعریف نگاشت احساسات و حلقه اصلی استخراج 


In [74]:

emotion_map = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

features = []   
labels = []     

# حلقه روی همه پوشه‌ها و فایل‌ها
for actor in actor_folders:
    actor_path = os.path.join(DATA_DIR, actor)
    if not os.path.isdir(actor_path):
        continue
    
    # فقط فایل‌های صوتی
    wav_files = [f for f in os.listdir(actor_path) if f.endswith('.wav')]
    
    for wav_file in wav_files:
        file_path = os.path.join(actor_path, wav_file)
        
        # استخراج کد احساس از عدد سوم نام فایل
        parts = wav_file.split('-')
        emotion_code = parts[2]
        label = emotion_map[emotion_code]
        
        # بارگذاری و استخراج MFCC
        y, sr = librosa.load(file_path, sr=None)
    
        # MFCC (۱۳ ویژگی)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)  # میانگین روی محور زمان
        
        # ZCR (نرخ عبور از صفر)
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)  
        
        # Spectral Centroid (مرکز ثقل طیفی)
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        cent_mean = np.mean(cent)  
        
        # ترکیب همه ویژگی‌ها در یک بردار ۱۵تایی
        combined = np.concatenate([mfcc_mean, [zcr_mean, cent_mean]])
        
        features.append(combined)
        labels.append(label)

X = np.array(features)
y = np.array(labels)

print(f"شکل ماتریس X: {X.shape}")
print(f"شکل بردار y: {y.shape}")
print(f"احساسات موجود در دیتاست: {np.unique(y)}")

شکل ماتریس X: (1440, 15)
شکل بردار y: (1440,)
احساسات موجود در دیتاست: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


#### گوش دادن به نمونه‌های واقعی از دیتاست


In [75]:
sample_actor_path = os.path.join(DATA_DIR, "Actor_01")

# لیستی از احساساتی که می‌خواهیم بشنویم
sample_emotions = [
    ('01', 'neutral',  'خنثی'),
    ('03', 'happy',    'شادی'),
    ('05', 'angry',    'خشم'),
    ('04', 'sad',      'غم')
]

print("🎧 گوش دادن به نمونه‌هایی از هر احساس\n")

for code, eng, persian in sample_emotions:
    filename = f"03-01-{code}-01-01-01-01.wav"
    filepath = os.path.join(sample_actor_path, filename)
    
    if os.path.exists(filepath):
        y_sample, sr_sample = librosa.load(filepath, sr=None)
        print(f"▶️  در حال پخش: {persian}")
        sd.play(y_sample, sr_sample)
        sd.wait()
        time.sleep(0.5)  # یک مکث کوتاه بین پخش‌ها
    else:
        print(f"فایل یافت نشد: {filepath}")

print("\n پایان پخش نمونه‌ها.")

🎧 گوش دادن به نمونه‌هایی از هر احساس

▶️  در حال پخش: خنثی
▶️  در حال پخش: شادی
▶️  در حال پخش: خشم
▶️  در حال پخش: غم

 پایان پخش نمونه‌ها.


### 4️⃣ StandardScaler نرمال‌سازی داده‌ها با 


In [76]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, 'scaler.pkl')

print("پس از نرمال‌سازی")
print(f"میانگین ۵ ویژگی اول\n {X_scaled.mean(axis=0)[:5]}\n")
print(f"انحراف معیار ۵ ویژگی اول\n {X_scaled.std(axis=0)[:5]}")

پس از نرمال‌سازی
میانگین ۵ ویژگی اول
 [ 3.43937841e-16 -6.78469626e-18  5.15791114e-17  8.21391566e-17
 -8.29583316e-17]

انحراف معیار ۵ ویژگی اول
 [1. 1. 1. 1. 1.]


### 5️⃣ تقسیم داده به آموزش و آزمون (Train-Test Split)


In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"تعداد داده‌های آموزش: {len(X_train)}")
print(f"تعداد داده‌های آزمون : {len(X_test)}")
print(f"\nچند برچسب از مجموعه آموزش: {y_train[:5]}")

تعداد داده‌های آموزش: 1152
تعداد داده‌های آزمون : 288

چند برچسب از مجموعه آموزش: ['happy' 'disgust' 'surprised' 'surprised' 'angry']


### 6️⃣ ذخیره‌سازی داده‌ها برای جلسه بعد


In [78]:
# ذخیره آرایه‌ها در یک فایل فشرده
np.savez('emotion_data.npz',
         X_train=X_train, X_test=X_test,
         y_train=y_train, y_test=y_test)

print("داده‌ها با موفقیت ذخیره شد. برای آموزش مدل در جلسه بعد آماده‌ایم")

داده‌ها با موفقیت ذخیره شد. برای آموزش مدل در جلسه بعد آماده‌ایم


### 🏁 جمع‌بندی پایانی


**امروز یاد گرفتیم که:**

 چطور یک دیتاست عظیم صوتی را با یک حلقه‌ی خودکار پردازش کنیم

از اسم رمزی فایل‌ها برچسب احساس را بیرون بکشیم

با میانگین‌گیری، صدا را به اعداد ثابت 15تایی تبدیل کنیم

و در نهایت با کمک کتابخانه ها داده‌هایمان را نرمال و به دو بخش آموزش و آزمون تقسیم کنیم


ما حالا یک ماتریس و یک بردار داریم که در جلسه‌ی بعدی، آن را به الگوریتم‌های ماشین لرنینگ می‌دهیم تا ببینیم کدام بهتر می‌تواند خشم و شادی را از هم تشخیص دهد

**!در ویدیوی بعدی: کارمان تکمیل شده و ماشین تشخیص احساسات و دروغ سنج ما آماده می شود**


In [79]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report

# # آموزش یک مدل جنگل تصادفی ساده فقط برای تست
# clf = RandomForestClassifier(n_estimators=100, random_state=42)
# clf.fit(X_train, y_train)

# # پیش‌بینی روی مجموعه آزمون
# y_pred = clf.predict(X_test)

# # نمایش دقت و گزارش
# print(f"✅ دقت روی داده‌های آزمون: {accuracy_score(y_test, y_pred):.2%}")
# print("\n📊 گزارش تفکیک هر احساس:")
# print(classification_report(y_test, y_pred))

In [80]:
# import sounddevice as sd
# import soundfile as sf

# # ضبط صدای زنده (۳ ثانیه)
# duration = 3
# sr = 44100
# print("🎙️ لطفاً یک جمله با یک احساس خاص بگویید (مثلاً عصبانی یا خوشحال)...")
# recording = sd.rec(int(sr * duration), samplerate=sr, channels=1)
# sd.wait()

# # ذخیره موقت
# sf.write("live_test.wav", recording, sr)

# y_live, _ = librosa.load("live_test.wav", sr=None)

# # استخراج MFCC (13 ویژگی)
# mfcc_live = librosa.feature.mfcc(y=y_live, sr=sr, n_mfcc=13)
# mfcc_live_mean = np.mean(mfcc_live, axis=1)

# # استخراج ZCR (نرخ عبور از صفر)
# zcr_live = librosa.feature.zero_crossing_rate(y_live)
# zcr_live_mean = np.mean(zcr_live)

# # استخراج Spectral Centroid
# cent_live = librosa.feature.spectral_centroid(y=y_live, sr=sr)
# cent_live_mean = np.mean(cent_live)

# # ترکیب همه ویژگی‌ها در یک بردار 15 تایی
# combined_live = np.concatenate([mfcc_live_mean, [zcr_live_mean, cent_live_mean]])

# # تبدیل به شکل (1,15) برای transform
# combined_live = combined_live.reshape(1, -1)

# # نرمال‌سازی با scaler (که قبلاً روی 15 ویژگی fit شده)
# combined_live_scaled = scaler.transform(combined_live)

# # پیش‌بینی
# predicted_emotion = clf.predict(combined_live_scaled)[0]
# print(f"🧠 تشخیص احساس توسط مدل: {predicted_emotion}")
